# 📝 Graph·Vector 하이브리드 검색 과제 LV1(기초)

교안 01의 벡터 검색, 출연 관계 확장, 전문 검색을 직접 구현합니다.

- 1~2번: 벡터 검색과 원문 중복 제거
- 3~5번: 확장 Cypher·VectorCypherRetriever·출연 경로 통합
- 6~8번: 확장 예산 비교·전문 검색·두 검색 결과 비교

<img src="images/05_lesson01_overview.png" width="1000">

**풀이 방법**: 교안 01 학습을 마친 뒤 새 커널에서 시작합니다. 각 문제의 빈 코드 셀에 함수·Cypher·실행 코드를 직접 작성하세요. 앞 문제의 결과는 다음 문제에서 이어 사용합니다. 힌트는 막혔을 때 펼치고, 각 주요 처리에는 짧은 주석을 붙이세요. 지문에 지정한 변수명·반환 키를 사용하고, 별도 지정이 없는 출력 형식은 자유입니다. 자가채점은 코드 결과를 확인하며, 검색 내용의 적합성은 원문을 읽고 판단합니다.

**데이터**: 기존 영화 그래프 171개 개체·253개 관계, 원문 38편·청크 571개를 사용합니다. 교안 01 본문에서 그래프와 두 검색 인덱스를 적재해 두세요. DB는 공유하며, 노트북 변수는 새로 만듭니다.

**모델 호출**: 정답 예시 기준으로 질문 임베딩·Jev 언어 판단·영어 검색어 변환 각 1회입니다. 문서 벡터는 DB에 적재된 값을 사용합니다. 검색 순위와 모델 문장은 고정하지 않으며, 마지막에는 원문과 관계 근거를 직접 읽습니다.


## 준비와 데이터 연결

패키지와 `.env` 설정은 README를 따릅니다. 노트북 파일이 있는 폴더에서 시작하세요. 필요한 라이브러리를 가져옵니다.


In [ ]:
# 검색기와 결과 형식을 직접 작성할 라이브러리를 가져옵니다.
import json
import os
import re
from pathlib import Path
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from pydantic import BaseModel, Field
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem


이 폴더의 설정으로 DB·모델을 연결하고 원문을 읽습니다. `movies`는 전체 패킷, `movies["relations"]`는 관계 목록, `documents_by_id`는 원문 ID로 제목·본문·출처를 찾는 사전입니다.


In [ ]:
# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def run_cypher(query, **params):
    """파라미터 쿼리의 조회 결과를 사전 목록으로 반환합니다."""
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


# 같은 자료의 원문과 관계 근거를 읽습니다.
movies = json.loads((material_dir / "data/movies_extraction_packet.json").read_text(encoding="utf-8"))
documents_by_id = movies["documents"]


## 1. 벡터 검색기를 만들고 청크를 찾습니다

**배경**: 우주선 사고를 설명한 원문을 한글 질문으로 찾습니다.

**요구사항**:

- `format_vector(record)`를 작성하세요. `record`의 node 항목이 검색된 청크입니다. `RetrieverResultItem`의 content에는 이 노드의 text를, metadata에는 노드의 id를 `chunk_id`, source_id를 `source_id`, record의 score를 `score`로 담습니다.
- `vector_retriever`는 `VectorRetriever`, 인덱스는 `movie_chunks`, embedder는 `embedding_model`, result_formatter는 위 함수로 설정하세요.
- `question`은 “달 탐사 임무 중 산소 탱크가 폭발한 우주선의 승무원들이 지구로 돌아오는 영화”로 정합니다.
- `embedding_model`로 질문을 한 번 임베딩해 `query_vector`에 저장하세요. 이 벡터로 최대 12개 청크를 검색한 반환 객체를 `vector_result`에 담습니다.

**확인 기준**: 질문 벡터는 768차원이며 검색 결과의 각 항목에 본문과 세 메타데이터가 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문 벡터를 한 번 만들어 뒤의 검색에서 재사용합니다.

세부구현:
1. formatter에서 Neo4j 노드의 속성과 검색 점수를 나누어 담습니다.
2. 검색기의 query_vector와 top_k로 검색 입력과 청크 수를 지정합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(vector_retriever, VectorRetriever)
assert len(query_vector) == 768
assert 0 < len(vector_result.items) <= 12
assert all(item.content and {"chunk_id", "source_id", "score"} <= item.metadata.keys() for item in vector_result.items)
print("✅ 1번 핵심 확인 완료!")


## 2. 청크 결과를 원문별로 합칩니다

**배경**: 같은 영화의 여러 청크가 검색되는 것은 자연스럽습니다. 이번에는 영화별 후보 목록을 만들기 위해 최고 점수 청크 한 건을 대표로 남깁니다.

**요구사항**:

- `vector_documents(query_vector, top_k=12)`를 작성하세요. `vector_retriever`로 주어진 query_vector와 top_k를 검색합니다. 결과의 items를 순서대로 읽고, source_id가 처음 나온 항목의 metadata를 그대로 담은 사전 목록을 반환합니다.
- 1번의 query_vector와 기본 top_k 12로 함수를 실행한 결과를 `seed_documents`에 담으세요. 제목은 documents_by_id에서 찾아 원문 ID·점수와 함께 출력합니다.

**확인 기준**: 영화별로 대표 후보 한 건을 남기며, 점수 내림차순 결과에서 각 원문이 처음 나온 청크와 점수를 유지합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 청크 ID와 원문 ID의 중복 단위가 다릅니다.

세부구현:
1. 검색 결과를 순서대로 읽고 이미 본 원문 ID를 기록합니다.
2. 처음 본 원문만 결과에 추가합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert seed_documents and len(seed_documents) == len({row["source_id"] for row in seed_documents})
assert all(row["source_id"] in documents_by_id for row in seed_documents)
assert all(row == next(item.metadata for item in vector_result.items if item.metadata["source_id"] == row["source_id"]) for row in seed_documents)
assert {row["source_id"] for row in seed_documents} == {item.metadata["source_id"] for item in vector_result.items}
print("✅ 2번 핵심 확인 완료!")


## 3. 배우를 거쳐 다른 영화로 확장하는 Cypher를 작성합니다

**배경**: 검색한 청크가 설명하는 영화에서 공통 배우가 출연한 다른 영화까지 이동합니다.

**요구사항**:

- 문자열 `expansion_query`에 Cypher를 작성하세요. `node`는 벡터 검색이 찾은 Chunk, `score`는 질문과 그 청크의 유사도입니다.
- FROM_DOCUMENT·ABOUT_MOVIE로 기준 영화를 찾고, ACTED_IN 두 관계로 공유 배우와 다른 영화를 찾습니다. 기준 영화 자신을 제외하고 개봉연도는 `$min_year` 이상으로 제한합니다.
- 각 시작 청크에서 후보 원문별로 `paths`를 만드세요. 각 항목은 `seed_title`(기준 영화 제목), `actor`(공유 배우 이름), `first_claim`(기준 영화 출연 관계의 claim_id), `second_claim`(후보 영화 출연 관계의 claim_id)을 담은 사전입니다. 같은 경로는 한 번만 남깁니다.
- 후보 원문 ID순으로 정렬한 뒤 **청크 하나당** `$max_per_seed`편만 남기세요.
- 반환 열은 `seed_chunk_id`(node의 id), `seed_score`(전달받은 score), `source_id`(후보 Document의 id), `title`(후보 원문 제목), `paths`입니다. 시작 청크와 후보 원문 조합마다 한 행을 반환합니다.

**확인 기준**: 다음 문제에서 실행해 관계 방향, 연도 제한, 청크별 확장 한도를 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 청크의 원문 연결을 먼저 찾고 영화 사이의 출연 경로를 이어 갑니다.

세부구현:
1. 같은 후보로 이어지는 배우 경로를 모은 뒤 후보를 정렬합니다.
2. CALL 서브쿼리 안에서 정렬과 LIMIT를 적용해 청크별 후보 수를 제한합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


## 4. VectorCypherRetriever에 확장 쿼리를 연결합니다

**배경**: 벡터 검색 직후 작성한 Cypher를 실행합니다.

**요구사항**:

- `format_expansion(record)`는 record.data()로 얻은 사전 전체를 metadata, 그 사전의 title 값을 content로 담은 `RetrieverResultItem`을 반환하도록 작성하세요.
- `graph_vector_retriever`를 `VectorCypherRetriever`로 만드세요. 인덱스·embedder는 1번과 같고 retrieval_query는 `expansion_query`, result_formatter는 위 함수입니다.
- `query_vector`를 사용해 top_k 2, query_params의 min_year 1994·max_per_seed 3으로 검색하고 `expanded_result`에 담으세요. 각 항목의 metadata를 표로 출력합니다.

**확인 기준**: 확장 결과에는 청크 ID, 기준 청크의 유사도, 후보 원문 ID와 두 출연 관계의 ID가 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색기가 node와 score를 Cypher로 전달합니다.

세부구현:
1. 결과 formatter에서 반환 열을 그대로 metadata로 보존합니다.
2. 추가 Cypher 파라미터는 query_params로 전달합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert isinstance(graph_vector_retriever, VectorCypherRetriever)
assert 0 < len(expanded_result.items) <= 6
assert len({item.metadata["seed_chunk_id"] for item in expanded_result.items}) <= 2
assert all(sum(other.metadata["seed_chunk_id"] == item.metadata["seed_chunk_id"] for other in expanded_result.items) <= 3 for item in expanded_result.items)
assert all(documents_by_id[item.metadata["source_id"]]["released"] >= 1994 for item in expanded_result.items)
assert all(path["seed_title"] != item.metadata["title"] for item in expanded_result.items for path in item.metadata["paths"])
assert all(
    any(relation["claim_id"] == path[key] and relation["relation"] == "ACTED_IN"
        and relation["source_record"]["source"] == path["actor"]
        and relation["source_record"]["target"] == title for relation in movies["relations"])
    for item in expanded_result.items for path in item.metadata["paths"]
    for key, title in [("first_claim", path["seed_title"]), ("second_claim", item.metadata["title"])])
assert all({"seed_chunk_id", "seed_score", "source_id", "title", "paths"} <= item.metadata.keys() for item in expanded_result.items)
assert all(item.metadata["paths"] and all({"seed_title", "actor", "first_claim", "second_claim"} <= path.keys() for path in item.metadata["paths"]) for item in expanded_result.items)
print("✅ 4번 핵심 확인 완료!")


## 5. 중복 영화의 출연 경로를 보존합니다

**배경**: 동일한 후보 영화에 도착한 여러 청크와 배우 경로를 합칩니다.

**요구사항**:

- `merge_expansions(result)`를 작성하세요. 입력은 4번의 검색 반환 객체입니다. source_id별로 합친 사전 목록을 반환하며, 각 사전에는 `source_id`, `title`, `seed_score`, `seed_chunks`, `paths`를 담습니다.
- seed_score는 같은 원문의 최댓값입니다. seed_chunks에는 원래 행들의 seed_chunk_id를, paths에는 출연 경로 사전을 중복 없이 모으세요. seed_score 내림차순, 동점이면 source_id 오름차순입니다.
- `expanded_result`를 합친 결과를 `expanded_documents`에 담고 출력하세요.

**확인 기준**: 원문마다 한 행이며, 원래 결과의 청크와 출연 경로가 빠지지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문서는 합치고 서로 다른 근거 경로는 남깁니다.

세부구현:
1. 원문 ID로 모으며 유사도는 최댓값을 유지합니다.
2. 청크와 경로는 중복 여부를 확인한 뒤 추가합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert len(expanded_documents) == len({item.metadata["source_id"] for item in expanded_result.items})
assert all(row["seed_score"] == max(item.metadata["seed_score"] for item in expanded_result.items if item.metadata["source_id"] == row["source_id"]) for row in expanded_documents)
assert all(set(row["seed_chunks"]) == {item.metadata["seed_chunk_id"] for item in expanded_result.items if item.metadata["source_id"] == row["source_id"]} for row in expanded_documents)
assert all({tuple(sorted(path.items())) for path in row["paths"]} == {tuple(sorted(path.items())) for item in expanded_result.items if item.metadata["source_id"] == row["source_id"] for path in item.metadata["paths"]} for row in expanded_documents)
assert all(len(row["paths"]) == len({tuple(sorted(path.items())) for path in row["paths"]}) for row in expanded_documents)
print("✅ 5번 핵심 확인 완료!")


## 6. 같은 질문으로 확장 예산을 비교합니다

**배경**: 검색 시작점과 관계 확장 한도가 후보 수에 미치는 영향을 확인합니다.

**요구사항**:

- `budget_results` 사전을 만드세요. 키 `(1, 2)`, `(3, 2)`, `(3, 4)`는 각각 top_k와 max_per_seed입니다. min_year는 1994로 고정하고 같은 query_vector를 재사용합니다.
- 각 값에는 `merge_expansions`로 합친 문서 목록을 담으세요.
- 각 설정의 고유 문서 수와 원문 ID를 출력합니다. (1, 2)에는 없고 (3, 4)에 있는 후보 중 하나의 원문을 읽으세요. 추가 후보가 없다면 그 사실을 기록합니다.

**확인 기준**: 세 설정 모두 이 자료에서 후보가 나오며, 질문 벡터·연도는 같고 시작 청크 수 또는 청크별 확장 수만 달라집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 한 번에 설정 하나가 달라지는 두 결과를 비교합니다.

세부구현:
1. 설정 쌍별로 검색하고 문서 중복을 합칩니다.
2. 원문 ID 집합으로 새 후보를 찾고 줄거리를 읽습니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert set(budget_results) == {(1, 2), (3, 2), (3, 4)}
assert all(rows for rows in budget_results.values())
assert all(len({chunk_id for row in rows for chunk_id in row["seed_chunks"]}) <= top_k for (top_k, max_per_seed), rows in budget_results.items())
assert all(len(rows) == len({row["source_id"] for row in rows}) for rows in budget_results.values())
assert all(len(rows) <= top_k * max_per_seed for (top_k, max_per_seed), rows in budget_results.items())
print("✅ 6번 핵심 확인 완료!")


**판단**: 후보 수가 늘면 질문에 맞는 결과도 반드시 늘어나는지, 읽은 원문을 근거로 1~2문장으로 적으세요.


*(여기에 판단과 근거를 서술하세요)*


## 7. 한글 질문을 Neo4j 전문 검색에 연결합니다

**배경**: 영어 원문에 등장하는 핵심어로 같은 질문을 검색합니다.

**요구사항**:

- `language_classifier`는 `TypeSafeClassifier`로 만들고, 모델은 환경변수 `TYPESAFE_MODEL`(기본값 `jev-1.13.0`)을 사용하세요. `language_question`은 `Choice`로 작성합니다. `ko`는 한글 및 영어 제목이 섞인 한글 문장, `en`은 한글 없는 영어 입력입니다.
- `EnglishKeywords`는 `keywords: list[str]` 필드를 가진 `BaseModel`로 작성하세요. `lexical_prompt`는 question을 받아 질문에 없는 영화명·배우명을 추가하지 않고 중복 없는 영어 단어 4~8개를 만들고, 각 항목에는 단어 하나씩 담도록 작성합니다. `lexical_chain`은 프롬프트와 `llm.with_structured_output(EnglishKeywords)`를 연결합니다.
- `fulltext_documents(query, top_k=12)`에서 Jev의 invoke에 state=query, questions={"language": language_question}을 전달하세요. 결과의 choices["language"].choice가 ko이면 lexical_chain에 question=query를 전달하고, keywords의 각 항목을 큰따옴표로 감싸 OR로 연결합니다. en이면 query를 그대로 사용합니다. 판단한 언어와 검색식을 출력하세요.
- `MATCH (node:Chunk)`에 `SEARCH node IN (FULLTEXT INDEX movie_fulltext FOR $search_text LIMIT $top_k) SCORE AS score`를 연결해 조회하세요. 검색된 node의 source_id·id를 각각 source_id·chunk_id로, 검색 점수를 score로 반환합니다. 점수 내림차순, 동점이면 chunk_id순입니다.
- 원문 ID별 최초 행만 남긴 사전 목록을 반환하세요. 1번의 question과 기본 top_k 12로 함수를 실행해 `fulltext_rows`에 담습니다.

**확인 기준**: 검색 결과의 원문 ID가 중복되지 않으며, 세 메타데이터를 담습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 영어 검색어 변환과 원문 중복 제거까지 직접 구현합니다.

세부구현:
1. Neo4j에는 한글 질문이 아니라 변환한 전문 검색식을 전달합니다.
2. 청크 점수순으로 원문별 최초 행을 남깁니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert 0 < len(fulltext_rows) <= 12
assert len(fulltext_rows) == len({row["source_id"] for row in fulltext_rows})
assert all({"source_id", "chunk_id", "score"} <= row.keys() and row["source_id"] in documents_by_id for row in fulltext_rows)
assert [row["score"] for row in fulltext_rows] == sorted([row["score"] for row in fulltext_rows], reverse=True)
print("✅ 7번 핵심 확인 완료!")


## 8. 검색 방식별로 찾은 원문을 비교합니다

**배경**: 단어 일치와 의미 유사도가 같은 영화와 서로 다른 영화를 찾는지 확인합니다.

**요구사항**:

- `dense_ids`, `fulltext_ids`에 각각 seed_documents와 fulltext_rows의 source_id 집합을 담으세요.
- 교집합은 `common_ids`, 전문 검색에만 있는 ID는 `fulltext_only`, 벡터 검색에만 있는 ID는 `dense_only`에 담고 출력하세요.
- 각 방식의 첫 문서 제목과 원문을 출력해 질문과 맞는지 읽으세요.

**확인 기준**: 공통·차이 집합의 의미가 맞고, 출력 원문을 근거로 두 방식을 비교합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색 방식이 달라도 비교 기준은 같은 원문 ID입니다.

세부구현:
1. 두 결과의 ID 집합에서 교집합과 양쪽 차집합을 구합니다.
2. 원래 점수의 크기로 두 검색 방식의 우열을 판단하지 않습니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


작성한 결과가 요구한 구조와 조건을 만족하는지 확인합니다.


In [ ]:
# [자가채점]
assert dense_ids == {row["source_id"] for row in seed_documents}
assert fulltext_ids == {row["source_id"] for row in fulltext_rows}
assert common_ids == dense_ids & fulltext_ids
assert fulltext_only == fulltext_ids - dense_ids
assert dense_only == dense_ids - fulltext_ids
print("✅ 8번 핵심 확인 완료!")


**판단**: 각 방식의 첫 원문이 우주선 사고 질문에 맞는지 근거 한 문장씩 적으세요.


*(여기에 판단과 근거를 서술하세요)*
